In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41590-021-00922-4"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41590_2021_922_MOESM2_ESM.xlsx"
table_name = "degs.txt" # local name # using this for paper_degs

## Read in file

In [9]:
df = pd.read_csv(table_name, delimiter="\t")

In [10]:
df

,Unnamed: 0,group,target,rank,nnz,nnz_frac,mean,log_fc,p_raw,p_corr
0,0,B,ENSG00000116251,94.0,30.0,1.000000,1.509471,0.517917,1.209840e-07,0.004076
1,1,B,ENSG00000142676,23.0,30.0,1.000000,2.732390,0.789916,3.597008e-10,0.000012
2,2,B,ENSG00000169442,77.0,29.0,0.966667,1.822012,1.093386,1.375236e-10,0.000005
3,3,B,ENSG00000142937,18.0,30.0,1.000000,2.819872,0.867924,1.682159e-09,0.000057
4,4,B,ENSG00000177954,1.0,30.0,1.000000,3.490398,0.829537,2.697463e-09,0.000091
...,...,...,...,...,...,...,...,...,...,...
2063,994,Neut,ENSG00000198763,7.0,118.0,0.991597,2.806447,0.526335,5.615098e-07,0.018920
2064,995,Neut,ENSG00000198804,1.0,117.0,0.983193,3.736855,0.708951,8.471008e-08,0.002854
2065,996,Neut,ENSG00000198712,10.0,117.0,0.983193,2.618784,0.525819,1.200207e-06,0.040440
2066,997,Neut,ENSG00000198840,15.0,116.0,0.974790,2.311032,0.533747,3.969406e-07,0.013375


In [14]:
df.drop(columns=["Unnamed: 0", "rank", "nnz", "nnz_frac", "mean"], inplace=True)

In [16]:
df.head()

,group,target,log_fc,p_raw,p_corr
0,B,ENSG00000116251,0.517917,1.209840e-07,0.004076
1,B,ENSG00000142676,0.789916,3.597008e-10,0.000012
2,B,ENSG00000169442,1.093386,1.375236e-10,0.000005
3,B,ENSG00000142937,0.867924,1.682159e-09,0.000057
4,B,ENSG00000177954,0.829537,2.697463e-09,0.000091


## Unfiltered

In [18]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None
unfiltered_df.rename(columns={"target": "feature_name"}, inplace=True)
unfiltered_df.rename(columns={"group": "cell_type_label"}, inplace=True)

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": "Jaitin Degs.txt",
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["log_fc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'B',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'ENSG00000116251',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'B',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'ENSG00000116251',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'Jaitin Degs.txt',
   'source_metrics': {'p_corr': 0.004076433712678,
    'log_fc': 0.517916533266448}}}]

## Save Unfiltered

In [19]:
with open("evidence_unfiltered_metrics.json", "w") as f:
    json.dump(result, f, indent = 4) 

## Perform the filtering

In [15]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [16]:

min_mean = 50
max_pval = 1e-10
min_lfc = 2
max_gene_shares = 10
max_per_celltype = 50

"""
min_mean = 0
max_pval = 1
min_lfc = 0
max_gene_shares = 100
max_per_celltype = 100
"""

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [17]:
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
25593,0.000000e+00,2.495286,0.166,0.003,0.000000e+00,B cell,IGHG3
25591,0.000000e+00,3.038060,0.173,0.003,0.000000e+00,B cell,IGHG1
25592,0.000000e+00,2.512809,0.193,0.001,0.000000e+00,B cell,IGLC3
23889,0.000000e+00,2.152270,0.283,0.003,0.000000e+00,Smooth muscle,THBS4
25588,0.000000e+00,3.649848,0.289,0.012,0.000000e+00,B cell,IGHA1
...,...,...,...,...,...,...,...
20174,0.000000e+00,3.062876,1.000,0.126,0.000000e+00,Preadipocyte,MGP
14585,0.000000e+00,2.210930,1.000,0.721,0.000000e+00,ncMo,SAT1
14589,0.000000e+00,2.106703,1.000,0.387,0.000000e+00,ncMo,COTL1
10648,1.009903e-241,2.551416,1.000,0.338,2.764206e-237,cDC1,HLA-DRB1


In [18]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
B cell           0.437200
Endothelial      0.796929
Mo-2             0.803667
Smooth muscle    0.823375
Myeloid-like     0.829450
Mo-1             0.857000
NK-like          0.859000
Neu              0.886000
LAM              0.886833
IM               0.899571
PVM              0.899618
cDC2A            0.917750
cDC2B            0.933059
ncMo             0.934471
APC              0.941500
mNK              0.950700
Preadipocyte     0.951235
cDC1             0.978538
ILCs             0.990000
Name: pct.1, dtype: float64

In [19]:
markers[col_ct].value_counts().sort_index()

cluster
APC               8
B cell           10
Endothelial      28
ILCs              1
IM                7
LAM               6
Mo-1              1
Mo-2              3
Myeloid-like     20
NK-like           2
Neu               6
PVM              34
Preadipocyte     17
Smooth muscle    16
cDC1             13
cDC2A             4
cDC2B            17
mNK              10
ncMo             17
Name: count, dtype: int64

## Find Ensembl ID


In [20]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [22]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [23]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [21]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = None

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG1',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'B cell',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG1',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale

## Save evidence

In [22]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 